In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

class ComprehensiveMatAnalysis:
    def __init__(self, mat_file_path):
        """
        深度分析MAT文件的数据分布和类别不平衡情况
        
        Args:
            mat_file_path: MAT文件路径
        """
        self.mat_file_path = mat_file_path
        self.data = None
        self.labels = None
        self.prob_idx = None
        self.analysis_results = {}
        
    def load_data(self):
        """加载MAT文件数据"""
        print("🔍 加载MAT文件数据...")
        
        with h5py.File(self.mat_file_path, 'r') as f:
            available_keys = list(f.keys())
            print(f"📁 MAT文件包含的键: {available_keys}")
            
            # 检查期望的键是否存在
            expected_keys = ['data', 'region', 'prob_idx']
            missing_keys = [key for key in expected_keys if key not in available_keys]
            if missing_keys:
                print(f"⚠️ 缺失的键: {missing_keys}")
                print(f"📋 可用的键: {available_keys}")
                raise ValueError(f"MAT文件缺少必需的键: {missing_keys}")
            
            # 加载数据 - 完全按照你原始代码的方式
            print("📊 加载 'data' 键...")
            self.data = np.array(f['data']).transpose()  # 特征数据
            print(f"   原始data形状: {np.array(f['data']).shape} -> 转置后: {self.data.shape}")
            
            print("📊 加载 'region' 键...")
            self.labels = np.array(f['region']).transpose()  # 标签数据
            print(f"   原始region形状: {np.array(f['region']).shape} -> 转置后: {self.labels.shape}")
            
            print("📊 加载 'prob_idx' 键...")
            self.prob_idx = np.array(f['prob_idx']).transpose()  # prob_idx数据
            print(f"   原始prob_idx形状: {np.array(f['prob_idx']).shape} -> 转置后: {self.prob_idx.shape}")
            
            # 验证数据一致性
            if self.data.shape[0] != self.labels.shape[0]:
                raise ValueError(f"数据和标签的样本数不匹配: data={self.data.shape[0]}, labels={self.labels.shape[0]}")
            
            if self.data.shape[0] != self.prob_idx.shape[0]:
                raise ValueError(f"数据和prob_idx的样本数不匹配: data={self.data.shape[0]}, prob_idx={self.prob_idx.shape[0]}")
            
            print(f"✅ 数据一致性检查通过")
            
        print(f"✅ 数据加载完成")
        print(f"   - 数据形状: {self.data.shape}")
        print(f"   - 标签形状: {self.labels.shape}")
        print(f"   - prob_idx形状: {self.prob_idx.shape}")
        
        return self
    
    def analyze_basic_statistics(self):
        """基础统计分析"""
        print("\n📊 基础统计分析...")
        
        results = {}
        
        # 数据基本信息
        results['data_info'] = {
            'total_samples': len(self.data),
            'total_features': self.data.shape[1],
            'data_type': str(self.data.dtype),
            'memory_usage_mb': self.data.nbytes / (1024**2)
        }
        
        # 标签信息分析
        if self.labels.ndim > 1 and self.labels.shape[1] > 1:
            # One-hot编码格式
            label_indices = np.argmax(self.labels, axis=1)
            results['label_format'] = 'one_hot'
            results['num_classes'] = self.labels.shape[1]
        else:
            # 直接标签格式
            label_indices = self.labels.flatten()
            results['label_format'] = 'direct'
            results['num_classes'] = len(np.unique(label_indices))
        
        results['label_indices'] = label_indices
        results['unique_labels'] = np.unique(label_indices)
        results['label_range'] = (int(np.min(label_indices)), int(np.max(label_indices)))
        
        # prob_idx分析
        unique_prob_idx = np.unique(self.prob_idx.flatten())
        results['prob_idx_info'] = {
            'unique_values': unique_prob_idx.tolist(),
            'counts': [int(np.sum(self.prob_idx == idx)) for idx in unique_prob_idx]
        }
        
        # 数据质量检查
        results['data_quality'] = {
            'has_nan': bool(np.isnan(self.data).any()),
            'has_inf': bool(np.isinf(self.data).any()),
            'nan_count': int(np.isnan(self.data).sum()),
            'inf_count': int(np.isinf(self.data).sum()),
            'zero_samples': int(np.sum(np.all(self.data == 0, axis=1))),
            'data_range': (float(np.min(self.data)), float(np.max(self.data))),
            'data_mean': float(np.mean(self.data)),
            'data_std': float(np.std(self.data))
        }
        
        self.analysis_results['basic_stats'] = results
        
        # 打印关键信息
        print(f"   📋 样本数量: {results['data_info']['total_samples']:,}")
        print(f"   📋 特征数量: {results['data_info']['total_features']:,}")
        print(f"   📋 类别数量: {results['num_classes']}")
        print(f"   📋 标签范围: {results['label_range']}")
        print(f"   📋 标签格式: {results['label_format']}")
        print(f"   📋 prob_idx取值: {results['prob_idx_info']['unique_values']}")
        print(f"   📋 数据范围: [{results['data_quality']['data_range'][0]:.3f}, {results['data_quality']['data_range'][1]:.3f}]")
        
        if results['data_quality']['has_nan'] or results['data_quality']['has_inf']:
            print(f"   ⚠️ 数据质量问题: NaN={results['data_quality']['nan_count']}, Inf={results['data_quality']['inf_count']}")
        
        return self
    
    def analyze_class_distribution(self, detailed=True):
        """类别分布分析"""
        print("\n📈 类别分布分析...")
        
        label_indices = self.analysis_results['basic_stats']['label_indices']
        
        # 计算类别分布
        class_counts = Counter(label_indices)
        total_samples = len(label_indices)
        
        # 基础分布统计
        distribution_stats = {
            'class_counts': dict(class_counts),
            'total_samples': total_samples,
            'num_classes_present': len(class_counts),
            'expected_num_classes': self.analysis_results['basic_stats']['num_classes']
        }
        
        # 计算类别统计
        counts_array = np.array(list(class_counts.values()))
        # Ensure counts_array is not empty before calculating min/max/imbalance ratio
        if len(counts_array) == 0: # All classes might be missing if labels are empty or problematic
            counts_array = np.array([0]) # Avoid errors with min/max on empty array

        distribution_stats.update({
            'mean_samples_per_class': float(np.mean(counts_array)),
            'std_samples_per_class': float(np.std(counts_array)),
            'min_samples': int(np.min(counts_array)) if len(counts_array) > 0 else 0,
            'max_samples': int(np.max(counts_array)) if len(counts_array) > 0 else 0,
            'median_samples': float(np.median(counts_array)) if len(counts_array) > 0 else 0.0,
            'imbalance_ratio': float(np.max(counts_array) / np.min(counts_array)) if np.min(counts_array) > 0 else float('inf')
        })
        
        # 识别缺失的类别
        all_possible_classes = set(range(self.analysis_results['basic_stats']['num_classes']))
        present_classes = set(class_counts.keys())
        missing_classes = sorted(list(all_possible_classes - present_classes))
        distribution_stats['missing_classes'] = missing_classes
        distribution_stats['missing_classes_count'] = len(missing_classes)
        
        # 类别分组分析
        percentiles = [10, 25, 50, 75, 90]
        percentile_values = np.percentile(counts_array, percentiles) if len(counts_array) > 0 else [0.0] * len(percentiles)
        distribution_stats['sample_count_percentiles'] = dict(zip(percentiles, percentile_values))
        
        # 不平衡程度分类
        imbalance_ratio = distribution_stats['imbalance_ratio']
        if imbalance_ratio <= 2:
            imbalance_level = "轻微不平衡"
        elif imbalance_ratio <= 5:
            imbalance_level = "中等不平衡"
        elif imbalance_ratio <= 10:
            imbalance_level = "严重不平衡"
        else:
            imbalance_level = "极度不平衡"
        
        distribution_stats['imbalance_level'] = imbalance_level
        
        # 分析prob_idx对类别分布的影响
        prob_idx_flat = self.prob_idx.flatten()
        prob_idx_analysis = {}
        
        for prob_val in np.unique(prob_idx_flat):
            mask = prob_idx_flat == prob_val
            subset_labels = label_indices[mask]
            subset_counts = Counter(subset_labels)
            
            prob_idx_analysis[int(prob_val)] = {
                'sample_count': int(np.sum(mask)),
                'unique_classes': len(subset_counts),
                'class_distribution': dict(subset_counts),
                'most_common_class': int(subset_counts.most_common(1)[0][0]) if subset_counts else None,
                'most_common_count': int(subset_counts.most_common(1)[0][1]) if subset_counts else 0
            }
        
        distribution_stats['prob_idx_analysis'] = prob_idx_analysis
        
        self.analysis_results['class_distribution'] = distribution_stats
        
        # 打印关键统计
        print(f"   📊 总样本数: {total_samples:,}")
        print(f"   📊 实际类别数: {distribution_stats['num_classes_present']}/{distribution_stats['expected_num_classes']}")
        print(f"   📊 缺失类别数: {distribution_stats['missing_classes_count']}")
        print(f"   📊 不平衡比例: {imbalance_ratio:.2f} ({imbalance_level})")
        print(f"   📊 样本数范围: {distribution_stats['min_samples']} - {distribution_stats['max_samples']}")
        print(f"   📊 平均每类样本: {distribution_stats['mean_samples_per_class']:.1f} ± {distribution_stats['std_samples_per_class']:.1f}")
        
        if missing_classes:
            print(f"   ⚠️ 缺失的类别: {missing_classes[:10]}{'...' if len(missing_classes) > 10 else ''}")
        
        return self
    
    def calculate_class_weights(self):
        """计算多种类别权重策略"""
        print("\n⚖️ 计算类别权重...")
        
        label_indices = self.analysis_results['basic_stats']['label_indices']
        class_counts = self.analysis_results['class_distribution']['class_counts']
        num_classes = self.analysis_results['basic_stats']['num_classes']
        total_samples = len(label_indices)
        
        weight_strategies = {}
        
        # 1. 平衡权重 (balanced)
        try:
            present_classes = np.array(list(class_counts.keys()))
            if len(present_classes) > 0 and len(label_indices) > 0 : # compute_class_weight requires non-empty inputs
                balanced_weights = compute_class_weight(
                    'balanced', 
                    classes=present_classes, 
                    y=label_indices
                )
                balanced_weight_dict = dict(zip(present_classes, balanced_weights))
                max_balanced_weight = max(balanced_weights) if len(balanced_weights) > 0 else 1.0
            else: # Handle cases with no present classes or no labels
                balanced_weight_dict = {}
                max_balanced_weight = 1.0

            missing_classes = self.analysis_results['class_distribution']['missing_classes']
            for missing_class in missing_classes:
                balanced_weight_dict[missing_class] = max_balanced_weight * 2
            
            weight_strategies['balanced'] = balanced_weight_dict
        except Exception as e:
            print(f"   ⚠️ 平衡权重计算失败: {e}")
            weight_strategies['balanced'] = {}
        
        # 2. 反频率权重 (inverse frequency)
        inverse_freq_weights = {}
        max_count_for_missing = max(class_counts.values()) if class_counts else 1
        for class_id in range(num_classes):
            count = class_counts.get(class_id, 0)
            if count > 0:
                inverse_freq_weights[class_id] = total_samples / (num_classes * count) if num_classes > 0 and count > 0 else 1.0
            else:
                inverse_freq_weights[class_id] = total_samples / (num_classes * 1) if num_classes > 0 else 1.0 # Assume at least 1 sample for missing
        
        weight_strategies['inverse_frequency'] = inverse_freq_weights
        
        # 3. 平方根权重 (sqrt balanced)
        sqrt_weights = {}
        total_sqrt_samples = np.sqrt(total_samples) if total_samples > 0 else 0.0
        for class_id in range(num_classes):
            count = class_counts.get(class_id, 0)
            if count > 0:
                sqrt_weights[class_id] = total_sqrt_samples / np.sqrt(count) if count > 0 else total_sqrt_samples
            else:
                sqrt_weights[class_id] = total_sqrt_samples # High weight for missing classes
        
        weight_strategies['sqrt_balanced'] = sqrt_weights
        
        # 4. 对数权重 (log balanced)
        log_weights = {}
        for class_id in range(num_classes):
            count = class_counts.get(class_id, 0)
            if count > 0 and total_samples > 0 and count < total_samples : # Avoid log(0) or log(negative)
                log_weights[class_id] = np.log(total_samples / count)
            elif count == total_samples : # If a class has all samples
                 log_weights[class_id] = np.log(1.0 + total_samples / count) # Small positive to avoid log(1)=0 weight
            else: # Missing classes
                log_weights[class_id] = np.log(total_samples if total_samples > 0 else 1.0) # High weight
        
        weight_strategies['log_balanced'] = log_weights
        
        # 5. 有效样本数权重 (effective number based)
        beta = 0.99
        effective_weights = {}
        for class_id in range(num_classes):
            count = class_counts.get(class_id, 0)
            if count > 0:
                effective_num = (1 - beta**count) / (1 - beta)
                effective_weights[class_id] = 1 / effective_num if effective_num > 0 else 1.0/(1-beta)
            else:
                effective_weights[class_id] = 1 / (1 - beta) # High weight
        
        weight_strategies['effective_number'] = effective_weights
        
        # 6. 分组权重 (prob_idx based)
        prob_idx_weights = {}
        prob_idx_analysis = self.analysis_results['class_distribution']['prob_idx_analysis']
        
        for class_id in range(num_classes):
            weights_by_prob = []
            
            for prob_val, prob_info in prob_idx_analysis.items():
                class_count_in_prob = prob_info['class_distribution'].get(class_id, 0)
                prob_total = prob_info['sample_count']
                num_unique_classes_in_prob = len(prob_info['class_distribution'])

                if class_count_in_prob > 0 and prob_total > 0 and num_unique_classes_in_prob > 0:
                    local_weight = prob_total / (num_unique_classes_in_prob * class_count_in_prob)
                    weights_by_prob.append(local_weight)
            
            if weights_by_prob:
                prob_idx_weights[class_id] = np.mean(weights_by_prob)
            else:
                prob_idx_weights[class_id] = 10.0 # High weight for classes not present in any prob_idx group
        
        weight_strategies['prob_idx_based'] = prob_idx_weights
        
        # 标准化所有权重策略
        for strategy_name, weights_dict in weight_strategies.items():
            if weights_dict: # Check if dictionary is not empty
                weight_array = np.array(list(weights_dict.values()))
                if len(weight_array) > 0 and np.mean(weight_array) != 0: # Avoid division by zero
                    normalized_weights = weight_array / np.mean(weight_array)
                    weight_strategies[strategy_name] = dict(zip(weights_dict.keys(), normalized_weights))
                elif len(weight_array) > 0 : # If mean is zero, assign uniform weights (or handle as error)
                    weight_strategies[strategy_name] = {k: 1.0 for k in weights_dict.keys()}


        self.analysis_results['weight_strategies'] = weight_strategies
        
        # 打印权重统计
        print(f"   ✅ 生成了 {len(weight_strategies)} 种权重策略")
        for strategy_name, weights in weight_strategies.items():
            if weights:
                weight_values = list(weights.values())
                print(f"   📊 {strategy_name}: 范围[{min(weight_values):.3f}, {max(weight_values):.3f}], "
                      f"均值={np.mean(weight_values):.3f}")
        
        return self
    
    def analyze_feature_distribution(self):
        """特征分布分析"""
        print("\n🔍 特征分布分析...")
        
        feature_stats = {}
        
        # 基础特征统计
        feature_stats['means'] = np.mean(self.data, axis=0)
        feature_stats['stds'] = np.std(self.data, axis=0)
        feature_stats['mins'] = np.min(self.data, axis=0)
        feature_stats['maxs'] = np.max(self.data, axis=0)
        feature_stats['medians'] = np.median(self.data, axis=0)
        
        # 特征质量检查
        feature_stats['zero_variance_features'] = np.where(feature_stats['stds'] == 0)[0].tolist()
        feature_stats['low_variance_features'] = np.where(feature_stats['stds'] < 0.01)[0].tolist()
        feature_stats['high_variance_features'] = np.where(feature_stats['stds'] > np.percentile(feature_stats['stds'], 95))[0].tolist() if len(feature_stats['stds']) > 0 else []

        # 特征相关性（采样分析以节省时间）
        if self.data.shape[1] > 1000:
            # 随机采样一部分特征进行相关性分析
            sample_indices = np.random.choice(self.data.shape[1], 1000, replace=False)
            sample_data = self.data[:, sample_indices]
        elif self.data.shape[1] > 1: # Need at least 2 features for correlation
            sample_data = self.data
            sample_indices = np.arange(self.data.shape[1])
        else: # Not enough features for correlation
            sample_data = None
            sample_indices = []

        high_corr_pairs = []
        if sample_data is not None and sample_data.shape[1] > 1:
            correlation_matrix = np.corrcoef(sample_data.T)
            if correlation_matrix.ndim == 2: # Ensure corrcoef didn't return scalar for 1 feature
                for i in range(len(sample_indices)):
                    for j in range(i + 1, len(sample_indices)):
                        if abs(correlation_matrix[i, j]) > 0.9:
                            high_corr_pairs.append((int(sample_indices[i]), int(sample_indices[j]), float(correlation_matrix[i, j])))
        
        feature_stats['high_correlation_pairs'] = high_corr_pairs[:50]  # 只保留前50个
        
        # 异常值检测
        if self.data.shape[1] > 0: # Check if features exist
            q25 = np.percentile(self.data, 25, axis=0)
            q75 = np.percentile(self.data, 75, axis=0)
            iqr = q75 - q25
            
            outlier_bounds_lower = q25 - 1.5 * iqr
            outlier_bounds_upper = q75 + 1.5 * iqr
            
            outliers_per_feature = []
            for i in range(self.data.shape[1]):
                outliers = np.sum((self.data[:, i] < outlier_bounds_lower[i]) | (self.data[:, i] > outlier_bounds_upper[i]))
                outliers_per_feature.append(int(outliers))
            
            feature_stats['outliers_per_feature'] = outliers_per_feature
            feature_stats['features_with_many_outliers'] = np.where(np.array(outliers_per_feature) > len(self.data) * 0.05)[0].tolist() if len(self.data) > 0 else []
        else:
            feature_stats['outliers_per_feature'] = []
            feature_stats['features_with_many_outliers'] = []


        self.analysis_results['feature_analysis'] = feature_stats
        
        print(f"   📊 零方差特征: {len(feature_stats['zero_variance_features'])}")
        print(f"   📊 低方差特征: {len(feature_stats['low_variance_features'])}")
        print(f"   📊 高相关特征对: {len(feature_stats['high_correlation_pairs'])}")
        print(f"   📊 异常值较多的特征: {len(feature_stats['features_with_many_outliers'])}")
        
        return self
    
    def generate_comprehensive_report(self, save_path="mat_analysis_report.txt"):
        """生成综合分析报告"""
        print(f"\n📝 生成综合分析报告: {save_path}")
        
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write("MAT文件深度数据分析报告\n")
            f.write("=" * 60 + "\n\n")
            f.write(f"文件路径: {self.mat_file_path}\n")
            f.write(f"分析时间: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            
            # 基础统计
            basic_stats = self.analysis_results['basic_stats']
            f.write("1. 基础数据统计\n")
            f.write("-" * 30 + "\n")
            f.write(f"总样本数: {basic_stats['data_info']['total_samples']:,}\n")
            f.write(f"特征维度: {basic_stats['data_info']['total_features']:,}\n")
            f.write(f"类别数量: {basic_stats['num_classes']}\n")
            f.write(f"标签格式: {basic_stats['label_format']}\n")
            f.write(f"标签范围: {basic_stats['label_range']}\n")
            f.write(f"内存占用: {basic_stats['data_info']['memory_usage_mb']:.2f} MB\n\n")
            
            # 数据质量
            quality = basic_stats['data_quality']
            f.write("2. 数据质量评估\n")
            f.write("-" * 30 + "\n")
            f.write(f"数据范围: [{quality['data_range'][0]:.6f}, {quality['data_range'][1]:.6f}]\n")
            f.write(f"数据均值: {quality['data_mean']:.6f}\n")
            f.write(f"数据标准差: {quality['data_std']:.6f}\n")
            f.write(f"NaN值数量: {quality['nan_count']}\n")
            f.write(f"无穷值数量: {quality['inf_count']}\n")
            f.write(f"全零样本数: {quality['zero_samples']}\n\n")
            
            # 类别分布分析
            class_dist = self.analysis_results['class_distribution']
            f.write("3. 类别分布分析\n")
            f.write("-" * 30 + "\n")
            f.write(f"实际类别数: {class_dist['num_classes_present']}/{class_dist['expected_num_classes']}\n")
            f.write(f"缺失类别数: {class_dist['missing_classes_count']}\n")
            f.write(f"不平衡比例: {class_dist.get('imbalance_ratio', 'N/A'):.2f} ({class_dist.get('imbalance_level', 'N/A')})\n") # Added .get for safety
            f.write(f"样本数统计:\n")
            f.write(f"  最少: {class_dist['min_samples']}\n")
            f.write(f"  最多: {class_dist['max_samples']}\n")
            f.write(f"  平均: {class_dist['mean_samples_per_class']:.1f}\n")
            f.write(f"  中位数: {class_dist['median_samples']:.1f}\n")
            f.write(f"  标准差: {class_dist['std_samples_per_class']:.1f}\n\n")
            
            # 权重推荐
            if 'weight_strategies' in self.analysis_results:
                weights = self.analysis_results['weight_strategies']
                f.write("4. 权重策略推荐\n")
                f.write("-" * 30 + "\n")
                
                # 根据不平衡程度推荐策略
                imbalance_ratio = class_dist.get('imbalance_ratio', float('inf')) # Default to high if not found
                recommended = "inverse_frequency" # Default recommendation
                if imbalance_ratio <= 2:
                    recommended = "balanced"
                    f.write("推荐策略: 平衡权重 (数据不平衡程度较轻)\n")
                elif imbalance_ratio <= 5:
                    recommended = "sqrt_balanced"
                    f.write("推荐策略: 平方根平衡权重 (中等不平衡)\n")
                elif imbalance_ratio <= 10:
                    recommended = "effective_number"
                    f.write("推荐策略: 有效样本数权重 (严重不平衡)\n")
                else:
                    recommended = "inverse_frequency"
                    f.write("推荐策略: 反频率权重 (极度不平衡)\n")
                
                f.write(f"\n各策略权重范围:\n")
                for strategy_name, strategy_weights in weights.items():
                    if strategy_weights: # Check if dict is not empty
                        values = list(strategy_weights.values())
                        if values: # Check if list of values is not empty
                            marker = "👉 " if strategy_name == recommended else "   "
                            f.write(f"{marker}{strategy_name}: [{min(values):.3f}, {max(values):.3f}], 均值={np.mean(values):.3f}\n")
                
                f.write(f"\n推荐权重详情 ({recommended}):\n")
                if recommended in weights and weights[recommended]:
                    sorted_weights = sorted(weights[recommended].items(), key=lambda x: x[1], reverse=True)
                    
                    f.write("最高权重类别 (前20):\n")
                    for i, (class_id, weight) in enumerate(sorted_weights[:20]):
                        count = class_dist['class_counts'].get(class_id, 0)
                        f.write(f"  类别{class_id}: 权重={weight:.4f}, 样本数={count}\n")
                    
                    if len(sorted_weights) > 20 : # Only show if there are more than 20
                        f.write("\n最低权重类别 (后10):\n")
                        for i, (class_id, weight) in enumerate(sorted_weights[-10:]):
                            count = class_dist['class_counts'].get(class_id, 0)
                            f.write(f"  类别{class_id}: 权重={weight:.4f}, 样本数={count}\n")
                f.write("\n")
            
            # prob_idx分析
            f.write("5. prob_idx分组分析\n")
            f.write("-" * 30 + "\n")
            prob_analysis = class_dist.get('prob_idx_analysis', {}) # Added .get for safety
            for prob_val, info in prob_analysis.items():
                f.write(f"prob_idx = {prob_val}:\n")
                f.write(f"  样本数: {info['sample_count']}\n")
                f.write(f"  类别数: {info['unique_classes']}\n")
                f.write(f"  最多类别: {info.get('most_common_class', 'N/A')} ({info.get('most_common_count', 0)}样本)\n")
                
                sorted_classes = sorted(info.get('class_distribution', {}).items(), key=lambda x: x[1], reverse=True)[:5]
                f.write(f"  主要类别: {', '.join([f'{c}({n})' for c, n in sorted_classes])}\n\n")
            
            # 特征分析
            if 'feature_analysis' in self.analysis_results:
                feature_analysis = self.analysis_results['feature_analysis']
                f.write("6. 特征分析\n")
                f.write("-" * 30 + "\n")
                f.write(f"零方差特征数: {len(feature_analysis.get('zero_variance_features', []))}\n") # Added .get for safety
                f.write(f"低方差特征数: {len(feature_analysis.get('low_variance_features', []))}\n")
                f.write(f"高相关特征对数: {len(feature_analysis.get('high_correlation_pairs', []))}\n")
                f.write(f"异常值较多的特征数: {len(feature_analysis.get('features_with_many_outliers', []))}\n")
                
                zero_var_feats = feature_analysis.get('zero_variance_features', [])
                if zero_var_feats:
                    f.write(f"\n零方差特征索引: {zero_var_feats[:20]}\n")

                high_corr_pairs_report = feature_analysis.get('high_correlation_pairs', [])
                if high_corr_pairs_report:
                    f.write(f"\n高相关特征对 (前10):\n")
                    for i, (f1, f2, corr) in enumerate(high_corr_pairs_report[:10]):
                        f.write(f"  特征{f1} - 特征{f2}: {corr:.4f}\n")
            
            # 训练建议
            f.write("\n7. 训练策略建议\n")
            f.write("-" * 30 + "\n")
            
            imbalance_ratio = class_dist.get('imbalance_ratio', float('inf'))
            missing_count = class_dist.get('missing_classes_count', 0)
            
            f.write("数据预处理建议:\n")
            if quality.get('has_nan', False) or quality.get('has_inf', False):
                f.write("  ⚠️ 处理NaN和无穷值\n")
            if quality.get('zero_samples', 0) > 0:
                f.write("  ⚠️ 移除或处理全零样本\n")
            
            f.write("\n类别权重建议:\n")
            if missing_count > 0:
                f.write(f"  ⚠️ {missing_count}个类别无样本，考虑:\n")
                f.write("     - 数据增强\n")
                f.write("     - 合并相似类别\n")
                f.write("     - 使用高权重策略\n")
            
            if imbalance_ratio > 10:
                f.write("  🎯 极度不平衡，强烈建议:\n")
                f.write("     - 使用反频率权重或有效样本数权重\n")
                f.write("     - 考虑focal loss\n")
                f.write("     - 数据重采样(SMOTE/ADASYN)\n")
            elif imbalance_ratio > 5:
                f.write("  📊 严重不平衡，建议:\n")
                f.write("     - 使用平方根平衡权重\n")
                f.write("     - 分层采样\n")
            elif imbalance_ratio > 2:
                f.write("  📈 中等不平衡，建议:\n")
                f.write("     - 使用标准平衡权重\n")
            else:
                f.write("  ✅ 数据相对平衡\n")
            
            f.write("\n模型训练建议:\n")
            f.write("  📚 批次采样策略:\n")
            if imbalance_ratio > 5:
                f.write("     - 使用BalancedBatchSampler\n")
                f.write("     - 每个批次确保类别平衡\n")
            
            f.write("  📊 验证策略:\n")
            f.write("     - 使用分层K折交叉验证\n")
            f.write("     - 监控每个类别的F1分数\n")
            f.write("     - 关注少数类别的召回率\n")
            
            f.write("  🎯 评估指标:\n")
            f.write("     - 主要指标: F1-macro, 平衡准确率\n")
            f.write("     - 辅助指标: F1-weighted, Cohen's Kappa\n")
            f.write("     - 避免单独使用准确率\n")
            
            if 'prob_idx_analysis' in class_dist:
                f.write("  🔄 prob_idx利用:\n")
                f.write("     - 考虑按prob_idx分组训练\n")
                f.write("     - 或者作为额外特征输入\n")
                f.write("     - 分析不同组的性能差异\n")
        
        print(f"✅ 分析报告已保存到: {save_path}")
        return self
    
    def visualize_data_distribution(self, save_dir="./analysis_plots/"):
        """生成数据分布可视化图表"""
        import os
        os.makedirs(save_dir, exist_ok=True)
        
        print(f"\n📊 生成可视化图表...")
        
        # Set font for English, DejaVu Sans is a good default if available
        plt.rcParams['font.sans-serif'] = ['DejaVu Sans'] 
        plt.rcParams['axes.unicode_minus'] = False # Ensure minus sign is displayed correctly
        
        # 1. 类别分布图 (Class Distribution Plot)
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        
        class_counts = self.analysis_results['class_distribution']['class_counts']
        classes = list(class_counts.keys())
        counts = list(class_counts.values())
        
        # Handle case with no classes/counts for plotting
        if not classes or not counts:
            print("   ⚠️ No class data to plot for class distribution.")
            # Optionally, create an empty plot or skip
            fig.text(0.5, 0.5, "No class data available for plotting.", ha='center', va='center', fontsize=12)
            plt.tight_layout()
            plt.savefig(os.path.join(save_dir, 'class_distribution_empty.png'), dpi=300, bbox_inches='tight')
            # plt.show() # Comment out if running in non-interactive environment
        else:
            # 类别样本数分布 (Sample Distribution per Class)
            axes[0, 0].bar(range(len(classes)), counts, alpha=0.7, color='skyblue')
            axes[0, 0].set_title('Sample Distribution per Class', fontsize=14, fontweight='bold')
            axes[0, 0].set_xlabel('Class ID')
            axes[0, 0].set_ylabel('Number of Samples')
            axes[0, 0].tick_params(axis='x', rotation=45)
            
            # 样本数分布直方图 (Histogram of Sample Counts)
            axes[0, 1].hist(counts, bins=30, alpha=0.7, color='lightgreen', edgecolor='black')
            axes[0, 1].set_title('Histogram of Sample Counts', fontsize=14, fontweight='bold')
            axes[0, 1].set_xlabel('Number of Samples')
            axes[0, 1].set_ylabel('Number of Classes')
            mean_counts = np.mean(counts) if counts else 0
            median_counts = np.median(counts) if counts else 0
            axes[0, 1].axvline(mean_counts, color='red', linestyle='--', label=f'Mean: {mean_counts:.1f}')
            axes[0, 1].axvline(median_counts, color='orange', linestyle='--', label=f'Median: {median_counts:.1f}')
            axes[0, 1].legend()
            
            # prob_idx分布 (prob_idx Sample Distribution)
            prob_idx_analysis = self.analysis_results['class_distribution']['prob_idx_analysis']
            prob_values = list(prob_idx_analysis.keys())
            prob_counts = [prob_idx_analysis[p]['sample_count'] for p in prob_values]
            
            if prob_counts: # Ensure there are counts to plot for pie chart
                axes[1, 0].pie(prob_counts, labels=[f'prob_idx={p}' for p in prob_values], 
                              autopct='%1.1f%%', startangle=90)
            axes[1, 0].set_title('prob_idx Sample Distribution', fontsize=14, fontweight='bold')
            
            # 累积分布 (Cumulative Distribution of Sample Counts)
            sorted_counts = sorted(counts)
            cumulative_pct = np.arange(1, len(sorted_counts) + 1) / len(sorted_counts) * 100 if len(sorted_counts) > 0 else []
            
            if list(cumulative_pct): # Check if cumulative_pct is not empty
                 axes[1, 1].plot(sorted_counts, cumulative_pct, marker='o', linewidth=2, markersize=4)
            axes[1, 1].set_title('Cumulative Distribution of Sample Counts', fontsize=14, fontweight='bold')
            axes[1, 1].set_xlabel('Number of Samples')
            axes[1, 1].set_ylabel('Cumulative Percentage')
            axes[1, 1].grid(True, alpha=0.3)
            
            # 添加重要统计信息 (Add important statistics)
            min_c = min(counts) if counts else 0
            max_c = max(counts) if counts else 0
            imbalance_r = (max_c / min_c) if min_c > 0 else float('inf')

            stats_text = f"""Data Statistics:
Total Classes: {len(classes)}
Sample Count Range: {min_c} - {max_c}
Imbalance Ratio: {imbalance_r:.2f}
Missing Classes: {self.analysis_results['class_distribution']['missing_classes_count']}"""
            
            fig.text(0.02, 0.02, stats_text, fontsize=10, 
                    bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
            
            plt.tight_layout()
            plt.savefig(os.path.join(save_dir, 'class_distribution.png'), dpi=300, bbox_inches='tight')
            # plt.show()

        # 2. 权重策略比较图 (Weight Strategy Comparison Plot)
        if 'weight_strategies' in self.analysis_results:
            fig, axes = plt.subplots(2, 3, figsize=(18, 12))
            axes = axes.flatten()
            
            weights_data = self.analysis_results['weight_strategies']
            strategy_names = list(weights_data.keys())
            
            plot_count = 0
            for i, strategy_name in enumerate(strategy_names):
                if plot_count >= 6:  # Max 6 strategies to display
                    break
                
                strategy_weights = weights_data[strategy_name]
                if strategy_weights: # Check if dict is not empty
                    # Ensure class_counts is available and not empty for filtering
                    if 'class_distribution' in self.analysis_results and \
                       'class_counts' in self.analysis_results['class_distribution'] and \
                       self.analysis_results['class_distribution']['class_counts']:
                        
                        class_counts_dict = self.analysis_results['class_distribution']['class_counts']
                        present_classes_ids = list(class_counts_dict.keys())
                        
                        # Filter weights for present classes and maintain order for plotting
                        # Create a list of (class_id, weight) for present classes
                        present_weights_map = {cls_id: strategy_weights.get(cls_id, 0) for cls_id in present_classes_ids}
                        
                        # Sort by class_id for consistent plotting order if desired, or plot as is
                        sorted_present_classes = sorted(present_weights_map.keys())
                        weights_to_plot = [present_weights_map[cls_id] for cls_id in sorted_present_classes]
                        class_labels_for_plot = [str(cls_id) for cls_id in sorted_present_classes]


                        if weights_to_plot: # Check if there are any weights to plot
                            axes[plot_count].bar(class_labels_for_plot, weights_to_plot, alpha=0.7)
                            axes[plot_count].set_title(f'{strategy_name} Weight Strategy', fontsize=12, fontweight='bold')
                            axes[plot_count].set_xlabel('Class ID')
                            axes[plot_count].set_ylabel('Weight Value')
                            axes[plot_count].tick_params(axis='x', rotation=45, labelsize=8) # Smaller labels if many classes
                            
                            min_w = min(weights_to_plot) if weights_to_plot else 0
                            max_w = max(weights_to_plot) if weights_to_plot else 0
                            axes[plot_count].text(0.02, 0.98, f'Range: [{min_w:.3f}, {max_w:.3f}]', 
                                       transform=axes[plot_count].transAxes, verticalalignment='top',
                                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                            plot_count += 1
                        else: # No weights for present classes
                            axes[plot_count].text(0.5, 0.5, "No weights for present classes", ha='center', va='center')
                            axes[plot_count].set_title(f'{strategy_name} Weight Strategy', fontsize=12, fontweight='bold')
                            plot_count +=1

                    else: # class_counts not available
                        axes[plot_count].text(0.5, 0.5, "Class counts data missing", ha='center', va='center')
                        axes[plot_count].set_title(f'{strategy_name} Weight Strategy', fontsize=12, fontweight='bold')
                        plot_count +=1
                else: # Strategy weights empty
                    axes[plot_count].text(0.5, 0.5, "No weights defined", ha='center', va='center')
                    axes[plot_count].set_title(f'{strategy_name} Weight Strategy', fontsize=12, fontweight='bold')
                    plot_count +=1


            # Hide unused subplots
            for i in range(plot_count, len(axes)):
                axes[i].set_visible(False)
            
            if plot_count > 0: # Only save if some plots were made
                plt.tight_layout()
                plt.savefig(os.path.join(save_dir, 'weight_strategies.png'), dpi=300, bbox_inches='tight')
            # plt.show()
        
        # 3. 特征分析图 (Feature Analysis Plot)
        if 'feature_analysis' in self.analysis_results:
            feature_analysis = self.analysis_results['feature_analysis']
            
            # Check if feature analysis results are meaningful for plotting
            if not feature_analysis.get('means', []).any() and not feature_analysis.get('stds', []).any():
                 print("   ⚠️ No feature data to plot for feature analysis.")
            else:
                fig, axes = plt.subplots(2, 2, figsize=(16, 12))
                
                # 特征均值分布 (Feature Mean Distribution)
                if feature_analysis['means'].any():
                    axes[0, 0].hist(feature_analysis['means'], bins=50, alpha=0.7, color='lightcoral')
                axes[0, 0].set_title('Feature Mean Distribution', fontsize=14, fontweight='bold')
                axes[0, 0].set_xlabel('Mean Value')
                axes[0, 0].set_ylabel('Number of Features')
                
                # 特征标准差分布 (Feature Standard Deviation Distribution)
                if feature_analysis['stds'].any():
                    axes[0, 1].hist(feature_analysis['stds'], bins=50, alpha=0.7, color='lightblue')
                    mean_stds = np.mean(feature_analysis["stds"]) if feature_analysis["stds"].any() else 0
                    axes[0, 1].axvline(mean_stds, color='red', linestyle='--', 
                                      label=f'Mean: {mean_stds:.4f}')
                    axes[0, 1].legend()
                axes[0, 1].set_title('Feature Standard Deviation Distribution', fontsize=14, fontweight='bold')
                axes[0, 1].set_xlabel('Standard Deviation')
                axes[0, 1].set_ylabel('Number of Features')
                
                # 异常值分布 (Distribution of Outlier Counts per Feature)
                outliers = feature_analysis.get('outliers_per_feature', [])
                if outliers:
                    axes[1, 0].hist(outliers, bins=30, alpha=0.7, color='lightyellow', edgecolor='black')
                axes[1, 0].set_title('Distribution of Outlier Counts per Feature', fontsize=14, fontweight='bold')
                axes[1, 0].set_xlabel('Number of Outliers')
                axes[1, 0].set_ylabel('Number of Features')
                
                # 特征重要性代理图（使用方差）(Proxy for Feature Importance (using Std Dev))
                feature_importance_proxy = feature_analysis.get('stds', np.array([]))
                if feature_importance_proxy.any():
                    top_n = min(20, len(feature_importance_proxy)) # Show top 20 or fewer if less features
                    if top_n > 0:
                        top_features_indices = np.argsort(feature_importance_proxy)[-top_n:]
                        top_feature_values = feature_importance_proxy[top_features_indices]
                        axes[1, 1].bar(range(top_n), top_feature_values, alpha=0.7, color='lightgreen')
                        axes[1, 1].set_xticks(range(top_n))
                        axes[1, 1].set_xticklabels([f'F{i}' for i in top_features_indices], rotation=45, ha="right")

                axes[1, 1].set_title(f'Top {top_n if "top_n" in locals() else 0} High Variance Features', fontsize=14, fontweight='bold')
                axes[1, 1].set_xlabel('Feature Index (Sorted by Std Dev)')
                axes[1, 1].set_ylabel('Standard Deviation')
                
                plt.tight_layout()
                plt.savefig(os.path.join(save_dir, 'feature_analysis.png'), dpi=300, bbox_inches='tight')
                # plt.show()
        
        # 4. prob_idx详细分析图 (prob_idx Detailed Analysis Plot)
        if 'class_distribution' in self.analysis_results and \
           'prob_idx_analysis' in self.analysis_results['class_distribution']:
            prob_idx_analysis = self.analysis_results['class_distribution']['prob_idx_analysis']

            if not prob_idx_analysis:
                print("   ⚠️ No prob_idx data to plot.")
            else:
                fig, axes = plt.subplots(1, 2, figsize=(16, 6))
                
                prob_values = list(prob_idx_analysis.keys())
                prob_samples = [prob_idx_analysis[p]['sample_count'] for p in prob_values]
                prob_classes = [prob_idx_analysis[p]['unique_classes'] for p in prob_values]
                
                x_pos = np.arange(len(prob_values))
                width = 0.35
                
                if prob_samples and prob_classes: # Ensure data exists
                    bars1 = axes[0].bar(x_pos - width/2, prob_samples, width, label='Sample Count', alpha=0.7, color='skyblue')
                    bars2 = axes[0].bar(x_pos + width/2, prob_classes, width, label='Class Count', alpha=0.7, color='lightcoral')
                    
                    axes[0].set_ylabel('Count')
                    axes[0].legend()
                    
                    # Add value labels to bars
                    for bar_group in [bars1, bars2]:
                        for bar in bar_group:
                            height = bar.get_height()
                            axes[0].text(bar.get_x() + bar.get_width()/2., height,
                                        f'{int(height)}', ha='center', va='bottom', fontsize=9)
                
                axes[0].set_title('Sample and Class Counts per prob_idx Group', fontsize=14, fontweight='bold')
                axes[0].set_xlabel('prob_idx')
                axes[0].set_xticks(x_pos)
                axes[0].set_xticklabels(prob_values)

                # 各prob_idx组内的类别分布差异 (Class Distribution Imbalance within prob_idx Groups)
                prob_diversities = []
                for prob_val in prob_values:
                    class_dist = prob_idx_analysis[prob_val]['class_distribution']
                    if class_dist:
                        counts_in_prob = list(class_dist.values())
                        mean_c = np.mean(counts_in_prob)
                        std_c = np.std(counts_in_prob)
                        diversity = std_c / mean_c if mean_c > 0 else 0  # Coefficient of Variation
                        prob_diversities.append(diversity)
                    else:
                        prob_diversities.append(0)
                
                if prob_diversities: # Ensure data exists
                    axes[1].bar([str(p) for p in prob_values], prob_diversities, alpha=0.7, color='lightgreen') # Use string labels for prob_values
                
                axes[1].set_title('Class Distribution Imbalance within prob_idx Groups', fontsize=14, fontweight='bold')
                axes[1].set_xlabel('prob_idx')
                axes[1].set_ylabel('Coefficient of Variation (Std/Mean)')
                
                plt.tight_layout()
                plt.savefig(os.path.join(save_dir, 'prob_idx_analysis.png'), dpi=300, bbox_inches='tight')
                # plt.show()
        else:
            print("   ⚠️ prob_idx analysis data not found for plotting.")

        print(f"✅ 所有可视化图表已保存到: {save_dir}")
        # Close all figures to free memory, especially in loops or scripts
        plt.close('all') 
        return self

    def get_recommended_weights(self, strategy='auto'):
        """获取推荐的类别权重"""
        if 'weight_strategies' not in self.analysis_results:
            print("⚠️ 请先运行calculate_class_weights()方法")
            return None
        
        weights = self.analysis_results['weight_strategies']
        imbalance_ratio = self.analysis_results['class_distribution'].get('imbalance_ratio', float('inf'))
        
        if strategy == 'auto':
            if imbalance_ratio <= 2:
                strategy = 'balanced'
            elif imbalance_ratio <= 5:
                strategy = 'sqrt_balanced'
            elif imbalance_ratio <= 10:
                strategy = 'effective_number'
            else:
                strategy = 'inverse_frequency'
            # Fallback if chosen strategy is not available or empty
            if not weights.get(strategy):
                 strategy = 'inverse_frequency' # A common robust default
                 if not weights.get(strategy): # If even inverse_frequency is missing, pick first available
                     available_strategies = [s for s, w in weights.items() if w]
                     if available_strategies:
                         strategy = available_strategies[0]
                     else: # No valid strategies
                         print(f"❌ 自动选择策略失败，且无可用策略.")
                         return None


        recommended_weights = weights.get(strategy, {})
        
        if recommended_weights:
            print(f"🎯 推荐使用: {strategy} 权重策略")
            print(f"   不平衡比例: {imbalance_ratio:.2f}")
            weight_values = list(recommended_weights.values())
            if weight_values: # Check if not empty
                 print(f"   权重范围: [{min(weight_values):.4f}, {max(weight_values):.4f}]")
            
            num_classes = self.analysis_results['basic_stats']['num_classes']
            weight_tensor = []
            for i in range(num_classes):
                weight_tensor.append(recommended_weights.get(i, 1.0)) # Default to 1.0 for missing classes in weights
            
            return {
                'strategy': strategy,
                'weights_dict': recommended_weights,
                'weights_tensor': weight_tensor,
                'imbalance_ratio': imbalance_ratio
            }
        else:
            print(f"❌ 策略 {strategy} 不可用或为空.")
            return None
    
    def export_weights_for_pytorch(self, strategy='auto', save_path='class_weights.py'):
        """导出适合PyTorch使用的权重代码"""
        weight_info = self.get_recommended_weights(strategy)
        
        if weight_info is None:
            print("⚠️ 无法导出权重: 未能获取推荐权重.")
            return
        
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write('"""\n')
            f.write('自动生成的类别权重配置\n')
            f.write(f'分析时间: {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
            f.write(f'推荐策略: {weight_info["strategy"]}\n')
            f.write(f'不平衡比例: {weight_info["imbalance_ratio"]:.2f}\n')
            f.write('"""\n\n')
            
            f.write('import torch\n')
            f.write('import numpy as np\n\n')
            
            f.write('# 类别权重张量 (适用于CrossEntropyLoss)\n')
            f.write(f'CLASS_WEIGHTS_TENSOR = torch.FloatTensor({weight_info["weights_tensor"]})\n\n')
            
            f.write('# 类别权重字典\n')
            f.write(f'CLASS_WEIGHTS_DICT = {weight_info["weights_dict"]}\n\n')
            
            f.write('# 配置信息\n')
            f.write('WEIGHT_CONFIG = {\n')
            f.write(f'    "strategy": "{weight_info["strategy"]}",\n')
            f.write(f'    "imbalance_ratio": {weight_info["imbalance_ratio"]:.4f},\n')
            f.write(f'    "num_classes": {len(weight_info["weights_tensor"])},\n')
            min_w = min(weight_info["weights_tensor"]) if weight_info["weights_tensor"] else 0.0
            max_w = max(weight_info["weights_tensor"]) if weight_info["weights_tensor"] else 0.0
            mean_w = np.mean(weight_info["weights_tensor"]) if weight_info["weights_tensor"] else 0.0
            f.write(f'    "min_weight": {min_w:.6f},\n')
            f.write(f'    "max_weight": {max_w:.6f},\n')
            f.write(f'    "mean_weight": {mean_w:.6f}\n')
            f.write('}\n\n')
            
            f.write('# 使用示例:\n')
            f.write('# import torch.nn as nn\n')
            f.write('# import torch.nn.functional as F\n')
            f.write('# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n')
            f.write('# criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS_TENSOR.to(device))\n')
            f.write('# \n')
            f.write('# 或者在自定义损失函数中使用:\n')
            f.write('# def weighted_cross_entropy(outputs, targets, weights=CLASS_WEIGHTS_TENSOR):\n')
            f.write('#     return F.cross_entropy(outputs, targets, weight=weights.to(outputs.device))\n')
        
        print(f"✅ 权重配置已导出到: {save_path}")
        return weight_info

# Using example and main function (remains unchanged)
def analyze_mat_file(mat_file_path, output_dir="./mat_analysis_output/"):
    """
    完整的MAT文件分析流程
    
    Args:
        mat_file_path: MAT文件路径
        output_dir: 输出目录
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    print("🚀 开始MAT文件深度分析...")
    print("=" * 60)
    
    # Create analyzer
    analyzer = ComprehensiveMatAnalysis(mat_file_path)
    
    # Execute complete analysis flow
    try:
        # 1. Data loading
        analyzer.load_data()
        
        # 2. Basic statistical analysis
        analyzer.analyze_basic_statistics()
        
        # 3. Class distribution analysis
        analyzer.analyze_class_distribution()
        
        # 4. Weight calculation
        analyzer.calculate_class_weights()
        
        # 5. Feature analysis
        analyzer.analyze_feature_distribution()
        
        # 6. Generate report
        report_path = os.path.join(output_dir, "comprehensive_analysis_report.txt")
        analyzer.generate_comprehensive_report(report_path)
        
        # 7. Generate visualization
        viz_dir = os.path.join(output_dir, "visualizations")
        analyzer.visualize_data_distribution(viz_dir) # This will now use English for plots
        
        # 8. Export weight configuration
        weights_path = os.path.join(output_dir, "recommended_class_weights.py")
        weight_info = analyzer.export_weights_for_pytorch(save_path=weights_path)
        
        print("\n" + "=" * 60)
        print("🎉 分析完成!")
        print("=" * 60)
        print(f"📁 输出目录: {output_dir}")
        print(f"📊 分析报告: {report_path}")
        print(f"📈 可视化图表: {viz_dir}")
        print(f"⚖️ 权重配置: {weights_path}")
        
        if weight_info:
            print(f"\n🎯 推荐权重策略: {weight_info['strategy']}")
            print(f"📊 不平衡比例: {weight_info['imbalance_ratio']:.2f}")
            if weight_info['weights_tensor']: # check if tensor is not empty
                print(f"⚖️ 权重范围: [{min(weight_info['weights_tensor']):.4f}, {max(weight_info['weights_tensor']):.4f}]")
        
        return analyzer
        
    except Exception as e:
        print(f"❌ 分析过程中出现错误: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

# Run analysis
if __name__ == "__main__":
    # Replace with your MAT file path
    # Create a dummy MAT file for testing if one is not available
    def create_dummy_mat(file_path="dummy_train.mat"):
        import os
        if os.path.exists(file_path):
            print(f"Dummy file {file_path} already exists. Using it.")
            return

        print(f"Creating dummy MAT file at {file_path} for testing...")
        num_samples = 1000
        num_features = 50
        num_classes = 10
        
        # Create sample data
        data = np.random.rand(num_features, num_samples).astype(np.float32) # Original shape (features, samples)
        
        # Create region (labels) - skewed distribution
        region_indices = np.random.choice(
            np.arange(num_classes), 
            size=num_samples, 
            p=np.concatenate([np.array([0.3, 0.2, 0.15, 0.1, 0.05]), np.full(num_classes-5, 0.2/(num_classes-5))]) # Skewed
        ).reshape(1, num_samples) # Original shape (1, samples)
        
        # Create prob_idx
        prob_idx = np.random.choice([1, 2, 3], size=num_samples).reshape(1, num_samples) # Original shape (1, samples)
        
        with h5py.File(file_path, 'w') as f:
            f.create_dataset('data', data=data)
            f.create_dataset('region', data=region_indices)
            f.create_dataset('prob_idx', data=prob_idx)
        print(f"Dummy MAT file {file_path} created.")

    # mat_file_path = "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat"
    # For demonstration, let's use a dummy file path.
    # You should replace this with your actual file path.
    dummy_file = "dummy_train_for_analysis.mat"
    create_dummy_mat(dummy_file) # Create if it doesn't exist
    mat_file_path = dummy_file   # Use the dummy file for the analysis
    
    output_directory = "./mat_analysis_output_english_plots/"

    # Execute analysis
    analyzer = analyze_mat_file(mat_file_path, output_dir=output_directory)
    
    if analyzer:
        print("\n🔍 快速访问分析结果:")
        print("analyzer.analysis_results['basic_stats']  # 基础统计")
        print("analyzer.analysis_results['class_distribution']  # 类别分布")
        print("analyzer.analysis_results['weight_strategies']  # 权重策略")
        print("analyzer.get_recommended_weights()  # 获取推荐权重")